# Anti-Churn Agent: Jev vs LLM Router Benchmark

#### Made by Fernando Cirone

## Hypothesis
Replacing the LLM router node in a LangGraph agent with Jev reduces latency and cost
significantly, without losing routing quality.

## Architecture
Both graphs are identical except for the router node:
- **Graph A (baseline):** Claude Sonnet as router → Claude Sonnet as specialist
- **Graph B (proposed):** Jev as router → Claude Sonnet as specialist

## What we measure
- Router latency and cost only (the variable)
- Specialist latency and cost (same in both, just for transparency)
- Total latency and cost (what matters in production)

## Index

0. Highlights and Notes
1. Step 1 - Setup (Claude and Jev clients)
2. Step 2 - State and Configuration (8 areas, test messages)
3. Step 3 - Router Functions (Claude router vs Jev router)
4. Step 4 - Specialist Node (identical in both graphs)
5. Step 5 - Build the Graphs (Graph A vs Graph B)
6. Step 6 - Run the Benchmark (5 messages)
7. Step 7 - Final Summary (5 messages)
8. Step 8 - Statistical Benchmark (50 planned, 48 completed)
9. Step 9 - Statistical Analysis (paired t-test, confidence interval)

## Highlights - 
#### **Replacing the Claude router with Jev in this LangGraph agent cut routing latency by 53% and routing cost by 97%, with 46 of 48 routes matching Claude's choice (N=48, p < 0.001).**

The only change between the two graphs is the router node. Same graph structure, same Claude specialist, same test messages.

| Metric | Claude Sonnet router | Jev router | Result |
|---|---|---|---|
| Router latency (mean) | 1,179 ms | 555 ms | 52.9% faster |
| Router cost per call | $0.000714 | $0.000021 | 97.0% cheaper |
| Agreement with Claude's route | n/a | 46 of 48 (95.8%) | 2 disagreements |
| Confidence score | not provided | mean 0.94, min 0.37 | returned natively |

- **Latency:** Jev saves 624 ms per routing decision (95% CI: 562 to 686 ms). Paired t-test: t = 20.22, p = 7.9e-25, N = 48.
- **Cost projection at 1M messages per month:** $714 with the Claude router vs $21 with Jev, about $693 saved on the router node alone.
- **Disagreements:** tests 26 and 28 were messages that could reasonably belong to either area. The lowest-confidence decision (0.37) was one of them.
- **Output format:** Jev returns a typed choice plus probabilities, so no text parsing is needed.

## Notes and Limitations

1. **Sample:** 48 of 50 planned messages completed (test 49 timed out on the Jev call, test 50 was not run). Messages are synthetic, in English, about 6 per area, and written for this demo. They are not real customer data.
2. **Agreement is not accuracy:** there is no human-labeled ground truth. A "match" means Jev chose the same area as Claude.
3. **Cost:** Claude cost uses exact token counts from the Anthropic API ($3 input / $15 output per 1M tokens). Jev cost is the exact figure reported by the Vercel AI Gateway (providerMetadata.gateway.cost, priced at $0.042 per 1M input tokens, output free). It was measured in a separate pass over the same messages, so it is not paired call by call with the latency measurements. An earlier version of this notebook estimated Jev cost from character counts and understated it (98.9% saving vs the real 97.0%).
4. **Latency setup:** calls were sequential from a single machine, Graph A always ran before Graph B (order not randomized), and Jev was reached through the Vercel AI Gateway, not TypeSafe's API directly.
5. **Baseline:** the Claude router is claude-sonnet-4-5 with a zero-shot prompt and max_tokens=10. A smaller model or prompt caching would likely narrow the gap. This was not tested.
6. **End-to-end:** the Claude specialist step takes about 5 to 14 seconds in both graphs and dominates total time, so the router saving is a small share of the full pipeline. End-to-end was not analyzed in the N=48 run, and specialist outputs vary between runs.
7. **Reliability:** Jev is in early access and was reached through the Vercel AI Gateway. Across the runs there was 1 error response (test 6, first attempt, cause not confirmed) and several 30 s read timeouts (test 49 in the main benchmark, tests 15, 18 and 19 in the cost pass). That is a small share of the calls, but in production you would need a timeout, retries and a fallback route.
8. **Confidence is a signal, not a guarantee:** some low-confidence routes still matched Claude (for example test 38, at 0.62).

## Step 1 — Setup
Install dependencies and initialize both clients:
- Claude Sonnet → used as router in Graph A and as specialist in both graphs
- Jev (via Vercel AI Gateway) → used as router in Graph B only

In [6]:
import os
import json
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from anthropic import Anthropic
from langgraph.graph import StateGraph, END
from typing import TypedDict

load_dotenv()

claude = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
CLAUDE_MODEL = os.getenv("LLM_MODEL", "claude-sonnet-4-5")

JEV_URL = "https://ai-gateway.vercel.sh/v4/ai/evaluation-model"
JEV_HEADERS = {
    "Authorization": f"Bearer {os.getenv('AI_GATEWAY_API_KEY')}",
    "Content-Type": "application/json",
    "ai-gateway-protocol-version": "0.0.1",
    "ai-evaluation-model-specification-version": "4",
    "ai-model-id": "typesafe-ai/jev",
}

print(f"   Claude model : {CLAUDE_MODEL}")
print(f"   Jev endpoint : {JEV_URL}")
print(f"   Jev key set  : {'yes' if os.getenv('AI_GATEWAY_API_KEY') else 'NO error'}")

   Claude model : claude-sonnet-4-5
   Jev endpoint : https://ai-gateway.vercel.sh/v4/ai/evaluation-model
   Jev key set  : yes


## Step 2 — State and Configuration
Define the graph state, the 8 knowledge base areas and the 5 test messages.
Note that the knowledge bases are simulated via system prompts — no vector store or RAG needed.
The focus is the router, not the final response.

In [8]:
class AgentState(TypedDict):
    message: str
    route: str
    response: str
    log: dict

# 8 areas with simulated knowledge bases (system prompts)
AREAS = {
    "billing": {
        "description": ["Duplicate charges, refunds, invoice issues, payment problems"],
        "system_prompt": "You are a billing specialist. Resolve the customer's financial issue empathetically and quickly to prevent churn. Offer immediate refund if necessary."
    },
    "technical_support": {
        "description": ["Bugs, errors, system outages, crashes, technical problems"],
        "system_prompt": "You are a technical support specialist. Solve the customer's technical problem with clear steps. Offer compensation if the issue is severe."
    },
    "logistics": {
        "description": ["Late deliveries, missing orders, tracking issues, returns"],
        "system_prompt": "You are a logistics specialist. Resolve delivery issues urgently and offer compensation for delays if necessary."
    },
    "product": {
        "description": ["Removed features, competitor comparisons, product dissatisfaction, suggestions"],
        "system_prompt": "You are a product specialist. Listen to the customer's frustration, explain the roadmap and offer alternatives to prevent churn."
    },
    "onboarding": {
        "description": ["Difficulty using the platform, setup problems, needs training"],
        "system_prompt": "You are an onboarding specialist. Help the customer get started quickly and offer a guided session if needed."
    },
    "cancellation": {
        "description": ["Direct cancellation request, wants to cancel subscription or service"],
        "system_prompt": "You are a retention specialist. This customer wants to cancel. Listen carefully, understand the root cause and make a strong offer to retain them."
    },
    "plans": {
        "description": ["Price complaints, upgrade or downgrade requests, plan comparisons"],
        "system_prompt": "You are a plans specialist. Address pricing concerns and offer a discount or better plan to retain the customer."
    },
    "security": {
        "description": ["Hacked account, privacy concerns, unauthorized access, data breach"],
        "system_prompt": "You are a security specialist. Treat this with maximum urgency. Secure the account immediately and reassure the customer about data safety."
    },
}

# 5 test messages covering different areas and churn signals
TEST_MESSAGES = [
    "I was charged twice this month and nobody is solving it. I'm about to cancel.",
    "The app crashes every time I try to generate a report.",
    "My order hasn't arrived in 15 days and the tracking hasn't updated at all.",
    "You removed the Slack integration I used every single day. I'm already looking at competitors.",
    "I want to cancel. The plan price went up 40 percent and I haven't seen any improvement.",
]

print("State and config ready")
print(f"Areas: {list(AREAS.keys())}")
print(f"Test messages: {len(TEST_MESSAGES)}")

State and config ready
Areas: ['billing', 'technical_support', 'logistics', 'product', 'onboarding', 'cancellation', 'plans', 'security']
Test messages: 5


## Step 3 — Router Functions
Define the two router functions: Claude as LLM router (Graph A) and Jev as router (Graph B).
These are the only nodes that differ between the two graphs.

In [11]:
def claude_router(state: AgentState) -> AgentState:
    """Graph A: uses Claude Sonnet as the router node"""

    areas_list = "\n".join([
        f'- "{k}": {v["description"][0]}'
        for k, v in AREAS.items()
    ])

    prompt = f"""You are a customer support router for an anti-churn system.
Given the customer message below, classify it into exactly one of these areas:

{areas_list}

Customer message: "{state['message']}"

Reply with ONLY the area key, nothing else. Example: billing"""

    start = time.time()
    response = claude.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": prompt}]
    )
    latency = (time.time() - start) * 1000

    route = response.content[0].text.strip().lower()

    # Fallback if Claude returns something unexpected
    if route not in AREAS:
        route = "cancellation"

    # Cost: Claude Sonnet input $3/1M, output $15/1M
    input_tokens  = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost = (input_tokens / 1_000_000 * 3.0) + (output_tokens / 1_000_000 * 15.0)

    print(f"   [Claude Router] → {route} | {latency:.0f}ms | ${cost:.6f}")

    return {
        **state,
        "route": route,
        "log": {
            "router": "claude",
            "router_latency_ms": latency,
            "router_cost_usd": cost,
            "router_input_tokens": input_tokens,
            "router_output_tokens": output_tokens,
        }
    }

print(f"   Graph A: Claude Sonnet ({CLAUDE_MODEL})")

   Graph A: Claude Sonnet (claude-sonnet-4-5)


In [21]:
def jev_router(state: AgentState) -> AgentState:
    """Graph B: uses Jev as the router node"""

    criteria = {k: v["description"] for k, v in AREAS.items()}

    payload = {
        "state": state["message"],
        "questions": {
            "route": {
                "type": "choice",
                "instructions": "Which support area should handle this customer message to prevent churn?",
                "criteria": criteria,
            }
        }
    }

    start = time.time()
    r = requests.post(JEV_URL, headers=JEV_HEADERS, json=payload, timeout=30)
    latency = (time.time() - start) * 1000

    data = r.json()

    # Handle error responses (rate limit, timeout, etc.)
    if "answers" not in data:
        print(f"   [Jev Router]    ⚠️ Error: {data.get('error', data)} | retrying in 5s...")
        time.sleep(5)
        start = time.time()
        r = requests.post(JEV_URL, headers=JEV_HEADERS, json=payload, timeout=30)
        latency = (time.time() - start) * 1000
        data = r.json()
        if "answers" not in data:
            print(f"   [Jev Router]    ❌ Retry failed, defaulting to cancellation")
            return {
                **state,
                "route": "cancellation",
                "log": {
                    "router": "jev",
                    "router_latency_ms": latency,
                    "router_cost_usd": 0,
                    "router_estimated_tokens": 0,
                    "router_confidence": 0,
                    "router_error": True,
                }
            }

    answer = data["answers"]["route"]
    route = answer["choice"]
    confidence = answer["probabilities"].get(route, 0)

    estimated_tokens = (len(state["message"]) + len(json.dumps(criteria))) / 4
    cost = (estimated_tokens / 1_000_000) * 0.042

    print(f"   [Jev Router]    → {route} | {latency:.0f}ms | ${cost:.6f} | confidence: {confidence:.2f}")

    return {
        **state,
        "route": route,
        "log": {
            "router": "jev",
            "router_latency_ms": latency,
            "router_cost_usd": cost,
            "router_estimated_tokens": int(estimated_tokens),
            "router_confidence": confidence,
        }
    }

print(f"   Graph B: Jev (typesafe-ai/jev)")

   Graph B: Jev (typesafe-ai/jev)


## Step 4 — Specialist Node
The specialist node is identical in both graphs.
It receives the route decision and responds using the appropriate system prompt.
This is where Claude Sonnet is used as the domain expert in both Graph A and Graph B.

In [22]:
def specialist_node(state: AgentState) -> AgentState:
    """Used in both graphs — Claude Sonnet responds as the domain specialist"""

    area = AREAS.get(state["route"], AREAS["cancellation"])
    system_prompt = area["system_prompt"]

    start = time.time()
    response = claude.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=int(os.getenv("LLM_MAX_TOKENS", 200)),
        system=system_prompt,
        messages=[{"role": "user", "content": state["message"]}]
    )
    latency = (time.time() - start) * 1000

    input_tokens  = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost = (input_tokens / 1_000_000 * 3.0) + (output_tokens / 1_000_000 * 15.0)

    print(f"   [Specialist: {state['route']}] | {latency:.0f}ms | ${cost:.6f}")

    # Merge specialist metrics into existing log
    updated_log = {
        **state["log"],
        "specialist_latency_ms": latency,
        "specialist_cost_usd": cost,
        "specialist_input_tokens": input_tokens,
        "specialist_output_tokens": output_tokens,
        "total_latency_ms": state["log"]["router_latency_ms"] + latency,
        "total_cost_usd": state["log"]["router_cost_usd"] + cost,
    }

    return {
        **state,
        "response": response.content[0].text.strip(),
        "log": updated_log,
    }



print(f"Model: {CLAUDE_MODEL}")
print(f"Max tokens: {os.getenv('LLM_MAX_TOKENS', 200)}")

Model: claude-sonnet-4-5
Max tokens: 2000


## Step 5 — Build the Graphs
Build Graph A (Claude router) and Graph B (Jev router).
The structure is identical — only the first node differs.

In [23]:
def get_route(state: AgentState) -> str:
    """Conditional edge: reads the route from state and directs to specialist"""
    return state["route"]


def build_graph(router_fn):
    """Builds a LangGraph with the given router function"""

    graph = StateGraph(AgentState)

    # Nodes
    graph.add_node("router", router_fn)
    graph.add_node("specialist", specialist_node)

    # Edges
    graph.set_entry_point("router")
    graph.add_conditional_edges(
        "router",
        get_route,
        {area: "specialist" for area in AREAS.keys()}
    )
    graph.add_edge("specialist", END)

    return graph.compile()


# Compile both graphs
graph_a = build_graph(claude_router)   # Graph A: Claude as router
graph_b = build_graph(jev_router)      # Graph B: Jev as router


print(f"   Graph A: Claude router → Claude specialist")
print(f"   Graph B: Jev router    → Claude specialist")
print(f"   Nodes: router → specialist → END")
print(f"   Conditional edges: {len(AREAS)} areas")

   Graph A: Claude router → Claude specialist
   Graph B: Jev router    → Claude specialist
   Nodes: router → specialist → END
   Conditional edges: 8 areas


## Step 6 — Run the Benchmark
Run all 5 test messages through both graphs and collect results.

In [15]:
results = []

for i, message in enumerate(TEST_MESSAGES):
    print(f"\n{'='*60}")
    print(f"Test {i+1}/5: {message[:60]}...")
    print(f"{'='*60}")

    # Graph A — Claude router
    print("\n▶ Graph A (Claude router):")
    result_a = graph_a.invoke({
        "message": message,
        "route": "",
        "response": "",
        "log": {}
    })

    # Graph B — Jev router
    print("\n▶ Graph B (Jev router):")
    result_b = graph_b.invoke({
        "message": message,
        "route": "",
        "response": "",
        "log": {}
    })

    results.append({
        "test": i + 1,
        "message": message[:60] + "...",
        # Router comparison
        "claude_route":            result_a["route"],
        "jev_route":               result_b["route"],
        "routes_match":            result_a["route"] == result_b["route"],
        # Router metrics
        "claude_router_ms":        result_a["log"]["router_latency_ms"],
        "jev_router_ms":           result_b["log"]["router_latency_ms"],
        "claude_router_cost_usd":  result_a["log"]["router_cost_usd"],
        "jev_router_cost_usd":     result_b["log"]["router_cost_usd"],
        # Specialist metrics (same model, shows natural variance)
        "claude_specialist_ms":    result_a["log"]["specialist_latency_ms"],
        "jev_specialist_ms":       result_b["log"]["specialist_latency_ms"],
        # Totals
        "claude_total_ms":         result_a["log"]["total_latency_ms"],
        "jev_total_ms":            result_b["log"]["total_latency_ms"],
        "claude_total_cost_usd":   result_a["log"]["total_cost_usd"],
        "jev_total_cost_usd":      result_b["log"]["total_cost_usd"],
        # Jev only
        "jev_confidence":          result_b["log"].get("router_confidence", 0),
    })

df = pd.DataFrame(results)
print("\n✅ Benchmark complete")
print(df[[
    "test", "claude_route", "jev_route", "routes_match",
    "claude_router_ms", "jev_router_ms",
    "claude_router_cost_usd", "jev_router_cost_usd"
]].to_string(index=False))


Test 1/5: I was charged twice this month and nobody is solving it. I'm...

▶ Graph A (Claude router):
   [Claude Router] → billing | 1534ms | $0.000711
   [Specialist: billing] | 7931ms | $0.004728

▶ Graph B (Jev router):
   [Jev Router]    → billing | 991ms | $0.000008 | confidence: 1.00
   [Specialist: billing] | 5890ms | $0.003408

Test 2/5: The app crashes every time I try to generate a report....

▶ Graph A (Claude router):
   [Claude Router] → technical_support | 1170ms | $0.000723
   [Specialist: technical_support] | 10174ms | $0.006138

▶ Graph B (Jev router):
   [Jev Router]    → technical_support | 567ms | $0.000008 | confidence: 1.00
   [Specialist: technical_support] | 9344ms | $0.005988

Test 3/5: My order hasn't arrived in 15 days and the tracking hasn't u...

▶ Graph A (Claude router):
   [Claude Router] → logistics | 1194ms | $0.000714
   [Specialist: logistics] | 11218ms | $0.004908

▶ Graph B (Jev router):
   [Jev Router]    → logistics | 497ms | $0.000008 | confide

## Step 7 — Final Summary
Calculate savings percentage on router latency and cost.
This is the core result of the benchmark.

In [16]:
# Router savings
avg_claude_router_ms   = df["claude_router_ms"].mean()
avg_jev_router_ms      = df["jev_router_ms"].mean()
avg_claude_router_cost = df["claude_router_cost_usd"].mean()
avg_jev_router_cost    = df["jev_router_cost_usd"].mean()

router_latency_savings = (1 - avg_jev_router_ms / avg_claude_router_ms) * 100
router_cost_savings    = (1 - avg_jev_router_cost / avg_claude_router_cost) * 100

# Total pipeline savings
avg_claude_total_ms   = df["claude_total_ms"].mean()
avg_jev_total_ms      = df["jev_total_ms"].mean()
avg_claude_total_cost = df["claude_total_cost_usd"].mean()
avg_jev_total_cost    = df["jev_total_cost_usd"].mean()

total_latency_savings = (1 - avg_jev_total_ms / avg_claude_total_ms) * 100
total_cost_savings    = (1 - avg_jev_total_cost / avg_claude_total_cost) * 100

print("=" * 60)
print("BENCHMARK RESULTS — Jev vs Claude as LangGraph Router")
print("=" * 60)

print(f"""
ROUTING NODE ONLY (the variable):
  Latency  → Claude: {avg_claude_router_ms:.0f}ms  |  Jev: {avg_jev_router_ms:.0f}ms
  Savings  → {router_latency_savings:.1f}% faster with Jev

  Cost     → Claude: ${avg_claude_router_cost:.6f}  |  Jev: ${avg_jev_router_cost:.6f}
  Savings  → {router_cost_savings:.1f}% cheaper with Jev

  Accuracy → {df["routes_match"].sum()}/{len(df)} routes matched between Claude and Jev
  Confidence (Jev) → avg {df["jev_confidence"].mean():.2f}

FULL PIPELINE (router + specialist):
  Latency  → Claude: {avg_claude_total_ms:.0f}ms  |  Jev: {avg_jev_total_ms:.0f}ms
  Savings  → {total_latency_savings:.1f}% faster end-to-end with Jev

  Cost     → Claude: ${avg_claude_total_cost:.6f}  |  Jev: ${avg_jev_total_cost:.6f}
  Savings  → {total_cost_savings:.1f}% cheaper end-to-end with Jev
""")

print("=" * 60)
print("CONCLUSION")
print("=" * 60)
print(f"""
Replacing the LLM router node with Jev in a LangGraph agent:
  ✅ {router_latency_savings:.0f}% faster routing
  ✅ {router_cost_savings:.0f}% cheaper routing
  ✅ {df["routes_match"].sum()}/{len(df)} routing decisions matched
  ✅ Jev returns calibrated confidence natively (Claude does not)
  ✅ Jev output is always typed — no JSON parsing needed

At scale (1M messages/month):
  Claude router cost → ${avg_claude_router_cost * 1_000_000:.2f}
  Jev router cost    → ${avg_jev_router_cost * 1_000_000:.2f}
  Monthly savings    → ${(avg_claude_router_cost - avg_jev_router_cost) * 1_000_000:.2f}
""")

df.to_csv("benchmark_antichurn.csv", index=False)
print("✅ Results saved to benchmark_antichurn.csv")

BENCHMARK RESULTS — Jev vs Claude as LangGraph Router

ROUTING NODE ONLY (the variable):
  Latency  → Claude: 1286ms  |  Jev: 650ms
  Savings  → 49.4% faster with Jev

  Cost     → Claude: $0.000720  |  Jev: $0.000008
  Savings  → 98.9% cheaper with Jev

  Accuracy → 5/5 routes matched between Claude and Jev
  Confidence (Jev) → avg 0.94

FULL PIPELINE (router + specialist):
  Latency  → Claude: 11364ms  |  Jev: 9444ms
  Savings  → 16.9% faster end-to-end with Jev

  Cost     → Claude: $0.006010  |  Jev: $0.004671
  Savings  → 22.3% cheaper end-to-end with Jev

CONCLUSION

Replacing the LLM router node with Jev in a LangGraph agent:
  ✅ 49% faster routing
  ✅ 99% cheaper routing
  ✅ 5/5 routing decisions matched
  ✅ Jev returns calibrated confidence natively (Claude does not)
  ✅ Jev output is always typed — no JSON parsing needed

At scale (1M messages/month):
  Claude router cost → $720.00
  Jev router cost    → $7.81
  Monthly savings    → $712.19

✅ Results saved to benchmark_antic

## Step 8 — Statistical Benchmark (N=50) - 50 planned, 48 completed
Run 50 test messages through both graphs to achieve statistical significance.
Messages cover all 8 areas with varying tone, urgency and churn signals.
With N=50 we can compute t-test, p-value and confidence intervals for latency.

⚠️ NOTE: Jev token count is estimated (chars / 4) because the Vercel AI Gateway
evaluation endpoint does not return real token usage. Claude token count is exact
via response.usage. Even with this underestimation, the cost difference remains
structurally large.

In [24]:
TEST_MESSAGES_50 = [
    # billing (7)
    "I was charged twice this month and nobody is solving it. I'm about to cancel.",
    "My invoice shows a charge for a plan I never subscribed to.",
    "You keep charging my old credit card even though I updated it weeks ago.",
    "I need a refund for the last 3 months. The service was barely working.",
    "There's a mysterious $49.99 charge on my statement I don't recognize.",
    "I downgraded my plan but I'm still being billed for the premium tier.",
    "Your billing support told me I'd get a refund 2 weeks ago. Still nothing.",

    # technical_support (7)
    "The app crashes every time I try to generate a report.",
    "I can't log in since yesterday. It just shows a white screen.",
    "The API keeps returning 500 errors. My team is blocked.",
    "Export to CSV has been broken for a week. We rely on this daily.",
    "The dashboard takes 45 seconds to load. It used to be instant.",
    "Two-factor authentication stopped working and I'm locked out.",
    "Every time I save a document the formatting gets completely destroyed.",

    # logistics (6)
    "My order hasn't arrived in 15 days and the tracking hasn't updated at all.",
    "I received the wrong item and need to return it immediately.",
    "The delivery was marked as complete but I never got the package.",
    "I've been waiting 3 weeks for a replacement. This is unacceptable.",
    "The courier left my package in the rain and everything is damaged.",
    "I need to change my delivery address but the order already shipped.",

    # product (6)
    "You removed the Slack integration I used every single day. I'm looking at competitors.",
    "The new UI update is terrible. Everything I need is now hidden behind 3 clicks.",
    "Your competitor just launched the exact feature I've been requesting for a year.",
    "The mobile app is missing half the features of the desktop version.",
    "You promised dark mode 6 months ago and it's still not here.",
    "The search function is useless. It never finds what I'm looking for.",

    # onboarding (6)
    "I signed up a week ago and still can't figure out how to set up my first project.",
    "Your documentation is outdated and doesn't match the current interface.",
    "I need someone to walk me through the initial setup. I'm completely lost.",
    "I watched all your tutorials but none of them cover the enterprise features I paid for.",
    "My team joined last month and half of them still don't understand how to use this.",
    "The onboarding wizard crashed halfway through and now I can't restart it.",

    # cancellation (6)
    "I want to cancel. The plan price went up 40% and I haven't seen any improvement.",
    "Please cancel my subscription effective immediately.",
    "I'm done. Cancel everything and delete my account.",
    "I want to cancel before the next billing cycle. How do I do that?",
    "Cancel my account. I've already moved my team to a competitor.",
    "I've asked to cancel three times now and I'm still being charged.",

    # plans (6)
    "The plan is way too expensive for what I get. Do you have anything cheaper?",
    "I want to downgrade but I can't find the option anywhere.",
    "My team grew from 5 to 50 people. What plan makes sense now?",
    "You raised the price but didn't add any new features. Not worth it.",
    "Is there a nonprofit discount? Our budget can't handle the current price.",
    "I only use 2 features out of 20. Why am I paying for a full plan?",

    # security (6)
    "Someone logged into my account from a country I've never been to.",
    "I got an email saying my password was changed but I didn't do it.",
    "I think my account was hacked. I see activity I don't recognize.",
    "My API keys were leaked on GitHub and I need to revoke everything now.",
    "A former employee still has access to our company account. Remove them immediately.",
    "I'm getting phishing emails that look exactly like your login page.",
]

print(f"{len(TEST_MESSAGES_50)} test messages ready")
print(f"Covering all {len(AREAS)} areas")

50 test messages ready
Covering all 8 areas


In [26]:
# Run benchmark
results_50 = []

for i, message in enumerate(TEST_MESSAGES_50):
    print(f"\n[{i+1}/{len(TEST_MESSAGES_50)}] {message[:55]}...")

    # Graph A — Claude router
    result_a = graph_a.invoke({
        "message": message, "route": "", "response": "", "log": {}
    })

    # Graph B — Jev router
    result_b = graph_b.invoke({
        "message": message, "route": "", "response": "", "log": {}
    })

    print(f"  Claude: {result_a['route']:20s} | {result_a['log']['router_latency_ms']:7.0f}ms | ${result_a['log']['router_cost_usd']:.6f}")
    print(f"  Jev:    {result_b['route']:20s} | {result_b['log']['router_latency_ms']:7.0f}ms | ${result_b['log']['router_cost_usd']:.6f} | conf: {result_b['log'].get('router_confidence',0):.2f}")

    results_50.append({
        "test": i + 1,
        "message": message[:55] + "...",
        "claude_route":           result_a["route"],
        "jev_route":              result_b["route"],
        "routes_match":           result_a["route"] == result_b["route"],
        "claude_router_ms":       result_a["log"]["router_latency_ms"],
        "jev_router_ms":          result_b["log"]["router_latency_ms"],
        "claude_router_cost":     result_a["log"]["router_cost_usd"],
        "jev_router_cost":        result_b["log"]["router_cost_usd"],
        "claude_total_ms":        result_a["log"]["total_latency_ms"],
        "jev_total_ms":           result_b["log"]["total_latency_ms"],
        "claude_total_cost":      result_a["log"]["total_cost_usd"],
        "jev_total_cost":         result_b["log"]["total_cost_usd"],
        "jev_confidence":         result_b["log"].get("router_confidence", 0),
    })

df50 = pd.DataFrame(results_50)
df50.to_csv("benchmark_50_antichurn.csv", index=False)
print(f"\n✅ Benchmark complete: {len(df50)} tests")
print(f"   Saved to benchmark_50_antichurn.csv")


[1/50] I was charged twice this month and nobody is solving it...
   [Claude Router] → billing | 1513ms | $0.000711
   [Specialist: billing] | 5974ms | $0.003318
   [Jev Router]    → billing | 1023ms | $0.000008 | confidence: 1.00
   [Specialist: billing] | 6736ms | $0.003948
  Claude: billing              |    1513ms | $0.000711
  Jev:    billing              |    1023ms | $0.000008 | conf: 1.00

[2/50] My invoice shows a charge for a plan I never subscribed...
   [Claude Router] → billing | 1235ms | $0.000699
   [Specialist: billing] | 6772ms | $0.003591
   [Jev Router]    → billing | 539ms | $0.000008 | confidence: 1.00
   [Specialist: billing] | 8036ms | $0.004326
  Claude: billing              |    1235ms | $0.000699
  Jev:    billing              |     539ms | $0.000008 | conf: 1.00

[3/50] You keep charging my old credit card even though I upda...
   [Claude Router] → billing | 1246ms | $0.000702
   [Specialist: billing] | 8922ms | $0.004404
   [Jev Router]    → billing | 487ms

ReadTimeout: HTTPSConnectionPool(host='ai-gateway.vercel.sh', port=443): Read timed out. (read timeout=30)

In [27]:
df50 = pd.DataFrame(results_50)
df50.to_csv("benchmark_50_antichurn.csv", index=False)
print(f"✅ {len(df50)} tests saved")

✅ 48 tests saved


## Step 9 — Statistical Analysis (N=48)
T-test, p-value, confidence intervals and final summary.

In [29]:
from scipy import stats

n = len(df50)
matches = df50["routes_match"].sum()

# ── Router latency t-test
t_stat, p_value = stats.ttest_rel(df50["claude_router_ms"], df50["jev_router_ms"])

claude_mean = df50["claude_router_ms"].mean()
jev_mean    = df50["jev_router_ms"].mean()
diff_mean   = (df50["claude_router_ms"] - df50["jev_router_ms"]).mean()
diff_std    = (df50["claude_router_ms"] - df50["jev_router_ms"]).std()
ci_95       = stats.t.interval(0.95, df=n-1, loc=diff_mean, scale=diff_std / (n**0.5))

latency_saving_pct = (1 - jev_mean / claude_mean) * 100

# ── Router cost
claude_cost_mean = df50["claude_router_cost"].mean()
jev_cost_mean    = df50["jev_router_cost"].mean()
cost_saving_pct  = (1 - jev_cost_mean / claude_cost_mean) * 100

# ── Jev confidence
conf_mean = df50["jev_confidence"].mean()
conf_min  = df50["jev_confidence"].min()

print("=" * 60)
print(f"STATISTICAL ANALYSIS — N={n}")
print("=" * 60)

print(f"""
ROUTER LATENCY:
  Claude mean:  {claude_mean:.0f}ms
  Jev mean:     {jev_mean:.0f}ms
  Saving:       {latency_saving_pct:.1f}%
  Difference:   {diff_mean:.0f}ms ± {diff_std:.0f}ms
  95% CI:       [{ci_95[0]:.0f}ms, {ci_95[1]:.0f}ms]
  t-statistic:  {t_stat:.2f}
  p-value:      {p_value:.2e}
  Significant:  {"✅ YES (p < 0.05)" if p_value < 0.05 else "❌ NO"}

ROUTER COST:
  Claude mean:  ${claude_cost_mean:.6f}
  Jev mean:     ${jev_cost_mean:.6f}
  Saving:       {cost_saving_pct:.1f}%

ROUTING AGREEMENT:
  Matched:      {matches}/{n} ({matches/n*100:.1f}%)
  Mismatched:   {n - matches}/{n}

JEV CONFIDENCE:
  Mean:         {conf_mean:.2f}
  Min:          {conf_min:.2f}

AT SCALE (1M messages/month):
  Claude router: ${claude_cost_mean * 1_000_000:.2f}
  Jev router:    ${jev_cost_mean * 1_000_000:.2f}
  Monthly saving: ${(claude_cost_mean - jev_cost_mean) * 1_000_000:.2f}
""")

# Show mismatches if any
mismatches = df50[~df50["routes_match"]]
if len(mismatches) > 0:
    print("MISMATCHED ROUTES:")
    for _, row in mismatches.iterrows():
        print(f"  Test {row['test']}: Claude={row['claude_route']}, Jev={row['jev_route']} (conf={row['jev_confidence']:.2f})")
        print(f"    → {row['message']}")

print("=" * 60)

STATISTICAL ANALYSIS — N=48

ROUTER LATENCY:
  Claude mean:  1179ms
  Jev mean:     555ms
  Saving:       52.9%
  Difference:   624ms ± 214ms
  95% CI:       s, 686ms]
  t-statistic:  20.22
  p-value:      7.94e-25
  Significant:  ✅ YES (p < 0.05)

ROUTER COST:
  Claude mean:  $0.000714
  Jev mean:     $0.000008
  Saving:       98.9%

ROUTING ACCURACY:
  Matched:      46/48 (95.8%)
  Mismatched:   2/48

JEV CONFIDENCE:
  Mean:         0.94
  Min:          0.37

AT SCALE (1M messages/month):
  Claude router: $714.06
  Jev router:    $7.72
  Monthly saving: $706.35

MISMATCHED ROUTES:
  Test 26: Claude=product, Jev=technical_support (conf=0.72)
    → The search function is useless. It never finds what I'm...
  Test 28: Claude=onboarding, Jev=technical_support (conf=0.37)
    → Your documentation is outdated and doesn't match the cu...


In [33]:
rows = []
first_usage = None

for t in df50["test"]:
    message = TEST_MESSAGES_50[t - 1]
    payload = {
        "state": message,
        "questions": {
            "route": {
                "type": "choice",
                "instructions": "Which support area should handle this customer message to prevent churn?",
                "criteria": {k: v["description"] for k, v in AREAS.items()},
            }
        },
    }

    try:
        r = requests.post(JEV_URL, headers=JEV_HEADERS, json=payload, timeout=30)
        data = r.json()
        cost = float(data["providerMetadata"]["gateway"]["cost"])
        if first_usage is None:
            first_usage = data.get("usage")
    except Exception as e:
        print(f"Test {t}: failed ({e})")
        cost = None

    rows.append({"test": t, "jev_real_cost": cost})
    time.sleep(0.3)

real = pd.DataFrame(rows)
df50 = df50.merge(real, on="test", how="left")

valid = df50.dropna(subset=["jev_real_cost"])
claude_cost = valid["claude_router_cost"].mean()
jev_est_cost = valid["jev_router_cost"].mean()
jev_real_cost = valid["jev_real_cost"].mean()

print("Usage field (first call):", first_usage)
print(f"\nCalls with real cost: {len(valid)}/{len(df50)}")
print(f"Claude router cost (exact):   ${claude_cost:.6f}")
print(f"Jev cost (old estimate):      ${jev_est_cost:.6f}")
print(f"Jev cost (real, from gateway): ${jev_real_cost:.6f}")
print(f"\nCost saving (real):     {(1 - jev_real_cost / claude_cost) * 100:.1f}%")
print(f"Cost saving (estimate): {(1 - jev_est_cost / claude_cost) * 100:.1f}%")
print(f"\nAt 1M messages/month:")
print(f"  Claude router: ${claude_cost * 1_000_000:.2f}")
print(f"  Jev router:    ${jev_real_cost * 1_000_000:.2f}")
print(f"  Monthly saving: ${(claude_cost - jev_real_cost) * 1_000_000:.2f}")

df50.to_csv("benchmark_50_antichurn.csv", index=False)

Test 15: failed (HTTPSConnectionPool(host='ai-gateway.vercel.sh', port=443): Read timed out. (read timeout=30))
Test 18: failed (HTTPSConnectionPool(host='ai-gateway.vercel.sh', port=443): Read timed out. (read timeout=30))
Test 19: failed (HTTPSConnectionPool(host='ai-gateway.vercel.sh', port=443): Read timed out. (read timeout=30))
Usage field (first call): {'inputTokens': 505, 'outputTokens': 82}

Calls with real cost: 45/48
Claude router cost (exact):   $0.000714
Jev cost (old estimate):      $0.000008
Jev cost (real, from gateway): $0.000021

Cost saving (real):     97.0%
Cost saving (estimate): 98.9%

At 1M messages/month:
  Claude router: $714.47
  Jev router:    $21.10
  Monthly saving: $693.37
